# 01 · Exploration et audit de qualité (PF.3)

Le livrable de l'étape PF.3 : ce notebook **exécuté**, et la section « Les données » de `docs/rapport.md`. Il audite le dataset que tu as téléchargé en PF.2 (`python scripts/download_data.py`), fait le découpage **une fois** (`data/splits.json`), puis explore l'entraînement.

**Deux temps, deux règles.** L'**audit structurel** regarde tout le dataset, sans lier les entrées à la cible : tailles, identifiants, doublons, valeurs manquantes, groupes, dates, répartition de la cible. Il sert à choisir le découpage (aléatoire, par groupes, temporel). L'**exploration statistique** (les relations entre les entrées et la cible, les figures qui inspirent des features) ne regarde **que l'entraînement** : ce que tu apprends en regardant le test finit toujours dans tes choix.

Chaque étape tourne dans `with step(...)` : tant qu'une fonction de ton paquet lève `NotImplementedError`, ou qu'une étape précédente n'a pas tourné, la cellule affiche ⏳ au lieu d'une erreur, et « Tout exécuter » va jusqu'au bout. Le notebook n'a pas de code d'apprentissage à lui : il appelle ton paquet (`src/projet/`). Ajoute tes propres cellules (figures, vérifications) là où le disent les `TODO`.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True   # True: configs/fast.yaml (CPU, < 10 min); False: configs/full.yaml (🚀 GPU Colab advised)
REPO_ON_DRIVE = "/content/drive/MyDrive/TODO-your-repository"   # on Colab only: where your repository is cloned

import os
import sys
from contextlib import contextmanager
from pathlib import Path

import pandas as pd

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    os.chdir(REPO_ON_DRIVE)

ROOT = Path.cwd().resolve()
while not (ROOT / "configs" / "fast.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "configs" / "fast.yaml").exists():
    raise FileNotFoundError("Projet introuvable : ouvre ce notebook depuis ton dépôt (celui qui a configs/fast.yaml).")
os.chdir(ROOT)                                    # the paths of the configuration are relative to the project
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))         # your package, even without pip install -e


class Pending(Exception):
    """An earlier step of the notebook did not run."""


def need(**values):
    """Raise Pending if one of the values is still None (the step that computes it is not done)."""
    missing = [name for name, value in values.items() if value is None]
    if missing:
        raise Pending("attends d'abord : " + ", ".join(missing))


@contextmanager
def step(title):
    """Run one step: a function not written yet, or a missing earlier step, prints ⏳ instead of an error."""
    try:
        yield
    except NotImplementedError as err:
        print(f"⏳ {title} : pas encore écrit ({err})")
    except Pending as err:
        print(f"⏳ {title} : {err}")


from projet.config import default_config_path, load_config  # noqa: E402

CONFIG_PATH = default_config_path(FAST_MODE)
config = load_config(CONFIG_PATH)
print(f"Projet : {ROOT.name} · mode {'rapide' if FAST_MODE else 'complet'} ({CONFIG_PATH.name}) · tâche : {config.task}")

## 1. Charger les données

`data.load_examples(config)` rend une ligne par exemple, avec une colonne `id` unique et stable, la cible `config.target`, et les entrées brutes (le contrat est dans sa docstring).

In [ ]:
df = None
with step("charger les données (data.load_examples)"):
    from projet.data import load_examples

    df = load_examples(config)
    print(f"{len(df)} exemples, {df.shape[1]} colonnes")
    display(df.head())

## 2. L'audit structurel (tout le dataset)

Les défauts du ch. 29 (🐛 29.13, 🏆 29.16) : identifiants répétés, doublons exacts et **quasi-doublons** (la même image recadrée, le même texte à la casse près, plusieurs lignes d'une même personne), valeurs manquantes codées par un nombre (`-1`, `999`, `""`), colonne constante, colonne qui trahit la cible, unités incohérentes, labels mal écrits, classe très rare. Note chaque défaut trouvé, la commande qui l'a montré et ce que tu en fais : c'est la section « Limites et biais » de ta data card.

In [ ]:
with step("audit structurel"):
    need(df=df)
    print("identifiants uniques :", df["id"].is_unique)
    print("doublons exacts (hors id) :", int(df.drop(columns="id").astype(str).duplicated().sum()))
    missing = df.isna().mean().sort_values(ascending=False)
    print("part de valeurs manquantes :", missing[missing > 0].round(3).to_dict() or "aucune")
    print("valeurs distinctes par colonne :", df.nunique().to_dict())
    if config.target in df:
        print("répartition de la cible :", df[config.target].value_counts(normalize=True).round(3).to_dict())
    # TODO PF.3: the quasi-duplicates of YOUR data, the missing values coded as numbers, the constant columns and the
    # columns that give the target away (29.12, 29.16); the groups (config.group_column) and the dates

## 3. Le découpage, une fois pour toutes

`data.make_splits` applique la règle que l'audit impose (par groupes s'il y a des quasi-doublons ou plusieurs exemples d'une même entité, temporel si le modèle servira dans le futur), puis `data.save_splits` écrit `data/splits.json` et rend son SHA-256. Recopie cette somme dans `docs/protocole.md` (« SHA-256 du découpage »), puis commite et tague le protocole (PF.4). Si le fichier existe déjà, cette cellule le relit : un découpage figé ne se refait pas.

In [ ]:
splits = None
with step("découpage (data.make_splits, data.save_splits)"):
    need(df=df)
    from projet.data import load_splits, make_splits, save_splits, sha256_file

    split_file = Path(config.split_file)
    if split_file.exists():
        splits = load_splits(split_file)
        print(f"ℹ️ {split_file} existe déjà : il est relu, jamais refait. SHA-256 du découpage : `{sha256_file(split_file)}`")
    else:
        splits = make_splits(df, config)
        digest = save_splits(splits, split_file)
        print(f"✅ {split_file} écrit. À recopier dans docs/protocole.md : SHA-256 du découpage : `{digest}`")
    print("tailles :", {name: len(ids) for name, ids in splits.items()})

## 4. Le décalage entre découpages

Des comptages et des résumés, sans modèle : la répartition de la cible dans chaque découpage, la moyenne des entrées numériques, la période ou les groupes couverts. Un écart ici (une classe absente du test, une période différente) se dit dans le rapport ; il peut aussi faire revoir le découpage, **avant** le tag du protocole.

In [ ]:
with step("décalage entre découpages"):
    need(df=df, splits=splits)
    which = pd.Series({str(i): name for name, ids in splits.items() for i in ids})
    part = df["id"].astype(str).map(which).rename("découpage")
    if config.target in df:
        display(pd.crosstab(part, df[config.target], normalize="index").round(3))
    numeric = df.select_dtypes("number").drop(columns=["id", config.target], errors="ignore")
    if not numeric.empty:
        display(numeric.groupby(part).mean().T.round(3))
    # TODO PF.3: for images or texts, a summary per split (sizes, brightness, length in tokens, sources, dates)

## 5. L'exploration de l'entraînement

Ici seulement, les relations entre les entrées et la cible : distributions par classe, corrélations, exemples de chaque classe, séries tracées. Enregistre les figures du rapport dans `figures/` (`plt.savefig("figures/....png", dpi=150, bbox_inches="tight")`).

In [ ]:
train = None
with step("exploration de l'entraînement"):
    need(df=df, splits=splits)
    from projet.data import select

    train = select(df, splits, "train")
    print(f"{len(train)} exemples d'entraînement")
    # TODO PF.3: your figures and your observations (matplotlib), on train only

## Ce que tu retiens

Recopie dans `docs/rapport.md` (« Les données ») et dans `data/card.md` (« Contenu », « Limites et biais ») :

- la taille réelle du dataset et des trois découpages, et la règle du découpage, justifiée par l'audit ;
- chaque défaut trouvé, la commande qui l'a montré, et ce que tu en as fait (corrigé, retiré, gardé et signalé) ;
- le décalage entre découpages ;
- deux ou trois observations de l'exploration qui orientent les modèles (déséquilibre, features fortes, saisonnalité).

**La suite (PF.4)** : remplis `docs/protocole.md`, puis fige-le **avant toute expérience** :

```bash
git add docs/protocole.md data/splits.json && git commit -m "Protocole d'évaluation" && git tag protocole && git push --tags
```